# Class 27: Orchestration with LangGraph
State, nodes/edges, conditional edges, human-in-the-loop। এই notebook Ollama ছাড়াই চলে (fake LLM); শেষে optional Ollama cell আছে।

## 1. State আর Reducer
Node শুধু বদলানো key ফেরত দেয়। `Annotated[..., operator.add]` মানে ওই key-তে নতুন মান জোড়া লাগবে, overwrite হবে না।

In [ ]:
import operator
from typing import Annotated, TypedDict
from langgraph.graph import StateGraph, START, END

class S(TypedDict, total=False):
    last: str                                   # reducer নেই: overwrite
    log: Annotated[list[str], operator.add]     # reducer: append

def a(s): return {"last": "a", "log": ["a ran"]}
def b(s): return {"last": "b", "log": ["b ran"]}

g = StateGraph(S)
g.add_node("a", a); g.add_node("b", b)
g.add_edge(START, "a"); g.add_edge("a", "b"); g.add_edge("b", END)
print(g.compile().invoke({"log": []}))   # last='b', log দুটোই আছে

## 2. Conditional Edge
Router function state দেখে পরের node-এর নাম ফেরত দেয়। এখানে একটা counter loop, `limit` পর্যন্ত।

In [ ]:
class L(TypedDict, total=False):
    n: int

def inc(s): return {"n": s.get("n", 0) + 1}
def route(s): return "inc" if s["n"] < 3 else END

g = StateGraph(L)
g.add_node("inc", inc)
g.add_edge(START, "inc")
g.add_conditional_edges("inc", route, ["inc", END])
print(g.compile().invoke({"n": 0}))   # {'n': 3}

## 3. interrupt + checkpointer
`interrupt()` graph থামায়; একই `thread_id`-তে `Command(resume=...)` দিলে চলতে থাকে।

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

class A(TypedDict, total=False):
    ok: bool

def ask(s):
    return {"ok": bool(interrupt("Approve?"))}

g = StateGraph(A)
g.add_node("ask", ask); g.add_edge(START, "ask"); g.add_edge("ask", END)
app = g.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "demo"}}
r = app.invoke({}, cfg)
print(r["__interrupt__"][0].value)               # 'Approve?'
print(app.invoke(Command(resume=True), cfg))     # {'ok': True}

## 4. Self-Correcting Writer (fake LLM)
Project-এর `src/` ফোল্ডার ব্যবহার করে। প্রথম উত্তর ইচ্ছাকৃত ভুল, দ্বিতীয়টা ঠিক; error দ্বিতীয় prompt-এ ফেরত যায়।

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../Class 1 Project"))
from src.graph import build_graph

BAD  = "```python\ndef add(a, b):\n    return a - b\n```"
GOOD = "```python\ndef add(a, b):\n    return a + b\n```"
replies = iter([BAD, GOOD]); prompts = []
def fake_llm(p): prompts.append(p); return next(replies)

g = build_graph(fake_llm, require_approval=False)
out = g.invoke({"task": "add two numbers", "tests": "assert add(1, 2) == 3",
                "attempts": 0, "history": []}, {"configurable": {"thread_id": "nb1"}})
print(out["history"]); print(out["status"])
print("--- 2nd prompt had the error:", "AssertionError" in prompts[1])

## 5. Human approval সহ
Approval চালু থাকলে test pass করার পর graph থামে।

In [ ]:
replies = iter([GOOD])
g = build_graph(lambda p: next(replies), require_approval=True)
cfg = {"configurable": {"thread_id": "nb2"}}
out = g.invoke({"task": "add", "tests": "assert add(1, 2) == 3", "attempts": 0, "history": []}, cfg)
print(out["__interrupt__"][0].value["code"])
from langgraph.types import Command
print(g.invoke(Command(resume=True), cfg)["status"])

## 6. Model Reference Table

| Model | Use Case | Advantage | Limitation |
| --- | --- | --- | --- |
| **`llama3.1:8b`** (default) | General-purpose, ছোট function | Instruction-following ভালো | কঠিন algorithm-এ ভুল, retry লাগে |
| `mistral:7b` | দ্রুত inference | হালকা | Code quality আলাদা সুবিধা দেয় না |
| `qwen2.5-coder:7b` | শুধু code | Code benchmark-এ সেরা | General কাজে দুর্বল |
| `llama3.2:3b` | কম RAM | খুব ছোট | বেশি ভুল |

নিচের cell Ollama চালু থাকলে real model দিয়ে চালায়, না থাকলে বার্তা দেখায়।

In [ ]:
from src.llm import ollama_llm, OllamaUnavailable
try:
    g = build_graph(ollama_llm, require_approval=False, max_attempts=3)
    out = g.invoke({"task": "Write is_palindrome(s) ignoring case and punctuation.",
                    "tests": "assert is_palindrome('A man, a plan, a canal: Panama')\nassert not is_palindrome('hi')",
                    "attempts": 0, "history": []}, {"configurable": {"thread_id": "nb3"}})
    print(out["history"], out["status"]); print(out["code"])
except OllamaUnavailable as e:
    print("Ollama পাওয়া যায়নি:", e)